In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df_analysis=pd.read_csv('Data/data.csv')
print(df_analysis.info())

In [ ]:
#data cleaning and preprocessing

# Remove entries which do not have age, jersey number and nationality 

df_analysis = df_analysis[df_analysis['Nationality'].notna()]
df_analysis = df_analysis[df_analysis['Age'].notna()]
df_analysis = df_analysis[df_analysis['Jersey Number'].notna()]
# Fill missing values with 0 for numerical columns and mode for categorical columns 
df_analysis[['Goals per match', 'Headed goals', 'Goals with right foot', 'Goals with left foot','Penalties scored','Freekicks scored','Shots','Shots on target', 'Shooting accuracy %','Hit woodwork','Big chances missed','Clean sheets','Goals conceded','Tackles','Tackle success %','Offsides']] = df_analysis[['Goals per match', 'Headed goals', 'Goals with right foot', 'Goals with left foot', 'Penalties scored','Freekicks scored', 'Shots', 'Shots on target', 'Shooting accuracy %','Hit woodwork','Big chances missed','Clean sheets','Goals conceded','Tackles','Tackle success %','Offsides']].fillna(0)
df_analysis['Nationality']= df_analysis['Nationality'].fillna(df_analysis['Nationality'].mode()[0])
df_analysis['Age']= df_analysis['Age'].fillna(df_analysis['Age'].median())
# cleaning the percentage sign
df_analysis['Cross accuracy %'] = df_analysis['Cross accuracy %'].str.replace(r'%', '').astype(float)
df_analysis['Shooting accuracy %'] = df_analysis['Shooting accuracy %'].str.replace(r'%', '').astype(float)
df_analysis['Tackle success %'] = df_analysis['Tackle success %'].str.replace(r'%', '').astype(float)


print(df_analysis['Name'].duplicated().sum()) # check for duplicate entries(should be 0 )


# check if there are any invalid values in the percentage columns (values less than 0 or greater than 100)

for col in ['Shooting accuracy %','Tackle success %','Cross accuracy %']:
    if col in df_analysis.columns:
        invalid_values = df_analysis[(df_analysis[col] < 0) | (df_analysis[col] > 100)][col]
        print(f"{col} - number of invalid values: {len(invalid_values)}")
        if len(invalid_values) > 0:
            print(invalid_values)

In [ ]:

#data cleaning for forwards
#filtering forwards with at least 1 appearance
forwards = df_analysis[df_analysis['Position'].str.contains('Forward')&(df_analysis['Appearances'] > 0)]
# Drop the specified columns from the forwards DataFrame
cols_to_drop = [
    'Clean sheets', 'Goals conceded', 'Tackles', 'Tackle success %', 
    'Last man tackles', 'Blocked shots', 'Interceptions', 'Clearances', 
    'Headed Clearance', 'Clearances off line', 'Own goals', 'Saves', 
    'Penalties saved', 'Punches', 'High Claims', 'Catches', 'Sweeper clearances', 'Throw outs', 'Goal Kicks','Through balls'	,'Accurate long balls'
]

forwards = forwards.drop(columns=cols_to_drop)

#feature engineering
 
forwards['shot_conversion'] = np.where(
    forwards['Shots'] > 0,
    forwards['Goals'] / forwards['Shots'],
    np.nan
)

forwards['shots_per_match'] =forwards['Shots'] / forwards['Appearances']

#goal per shot on target
forwards['goals_per_shot_on_target'] = np.where(
    forwards['Shots on target'] > 0,
    forwards['Goals'] / forwards['Shots on target'],
    np.nan
)

forwards.describe()


In [ ]:
def the_top_player(df, column):
    top_value = df[column].max()
    players = df[df[column] == top_value]
    return list(zip(players['Name'], players[column]))

def the_bottom_player(df, column):
    bottom_value = df[column].min()
    players = df[df[column] == bottom_value]
    return list(zip(players['Name'], players[column]))

In [ ]:
#Descriptive analysis of forwards

#1-the best players for forwards 
def the_best_forward_players():
    #the top goals
    top_goals = the_top_player(forwards, 'Goals')
    #Player with Fewest Offsides
    bottom_offside = the_bottom_player(forwards, 'Offsides')
    #the Top Shot Conversion Rate
    top_shot_conversion = the_top_player(forwards, 'shot_conversion')
    #the Top Shooting Accuracy
    top_shooting_accuracy = the_top_player(forwards, 'Shooting accuracy %')
    #the Top goals per shot on target
    top_goals_per_match = the_top_player(forwards, 'goals_per_shot_on_target')
    #the tope right foot scorrer
    top_right_foot_scorrer = the_top_player(forwards, 'Goals with right foot')
    #the top left foot scorrer
    top_left_foot_scorrer = the_top_player(forwards, 'Goals with left foot')
    #the top assester
    top_assister = the_top_player(forwards, 'Assists')
    #the top shots per match
    top_shots_per_match = the_top_player(forwards, 'shots_per_match')
    #the top goals per shot on target
    top_goals_per_shot_on_target = the_top_player(forwards, 'goals_per_shot_on_target')
    print(
        f"Top Goals: {top_goals}\n"
        f"Top Assister: {top_assister}\n"
        f"Fewest Offsides: {bottom_offside}\n"
        f"Top Shot Conversion: {top_shot_conversion}\n"
        f"Top Shooting Accuracy: {top_shooting_accuracy}\n"
        f"Top Goals per Match: {top_goals_per_match}\n"
        f"Top Right Foot Scorer: {top_right_foot_scorrer}\n"
        f"Top Left Foot Scorer: {top_left_foot_scorrer}\n"
        f"Top Shots per Match: {top_shots_per_match}\n"
        f"Top Goals per Shot on Target: {top_goals_per_shot_on_target}\n"
    )
the_best_forward_players()

In [ ]:
sns.set(style="whitegrid")
club_goals = forwards.groupby('Club')['Goals'].sum().sort_values(ascending=False).head(10)
top_scorer_per_club = forwards.loc[forwards.groupby('Club')['Goals'].idxmax()][['Club', 'Name', 'Goals']]
top_scorer_per_club = top_scorer_per_club.sort_values(by='Goals', ascending=False).head(10)
club_assists = forwards.groupby('Club')['Assists'].sum().sort_values(ascending=False).head(10)
top_assist_players = forwards.sort_values(by='Assists', ascending=False).head(10)

plt.figure(figsize=(18, 10))

# plot 1
plt.subplot(2, 2, 1)
ax1 = sns.barplot(x=club_goals.index, y=club_goals.values)
plt.title('Top Clubs by Goals')
plt.xticks(rotation=45)

for i, v in enumerate(club_goals.values):
    ax1.text(i, v, str(v), ha='center', va='bottom')

# plot 2
plt.subplot(2, 2, 2)
ax2 = sns.barplot(x=top_scorer_per_club['Name'], y=top_scorer_per_club['Goals'])
plt.title('Top Scorer in Each Club')
plt.xticks(rotation=45)

for i, v in enumerate(top_scorer_per_club['Goals']):
    ax2.text(i, v, str(v), ha='center', va='bottom')

#plot 3 
plt.subplot(2, 2, 3)
ax3 = sns.barplot(x=club_assists.index, y=club_assists.values)
plt.title('Top Clubs by Assists')
plt.xticks(rotation=45)

for i, v in enumerate(club_assists.values):
    ax3.text(i, v, str(v), ha='center', va='bottom')

#plot 4   
plt.subplot(2, 2, 4)
ax4 = sns.barplot(x=top_assist_players['Name'], y=top_assist_players['Assists'])
plt.title('Top Players by Assists')
plt.xticks(rotation=45)

for i, v in enumerate(top_assist_players['Assists']):
    ax4.text(i, v, str(v), ha='center', va='bottom')

plt.tight_layout()
plt.show()

In [ ]:
#data cleaning for midfielders
#filtering midfielders with at least 1 appearance
midfielders = df_analysis[df_analysis['Position'].str.contains('Midfielder')&(df_analysis['Appearances'] > 0)]
# Drop the specified columns from the midfielders DataFrame
cols_to_drop_midfielder = [
    'Clean sheets','Goals conceded','Tackles','Tackle success %','Last man tackles',
    'Blocked shots','Interceptions','Clearances','Headed Clearance','Clearances off line',
    'Recoveries','Duels won','Duels lost','Successful 50/50s','Aerial battles won','Aerial battles lost',
    'Own goals','Errors leading to goal','Saves','Penalties saved','Punches','High Claims','Catches',
    'Sweeper clearances','Throw outs','Goal Kicks'
]

midfielders = midfielders.drop(columns=cols_to_drop_midfielder)
midfielders.describe()

#feature engineering for midfielders
midfielders['passes_per_match'] = midfielders['Passes'] / midfielders['Appearances']
midfielders['assists_per_match'] = midfielders['Assists'] / midfielders['Appearances']
midfielders['big_chances_created_per_match'] = midfielders['Big chances created'] / midfielders['Appearances']
midfielders['shots_per_match'] = midfielders['Shots'] / midfielders['Appearances']

midfielders['shot_conversion'] = np.where(
    midfielders['Shots'] > 0,
    midfielders['Goals'] / midfielders['Shots'],
    np.nan
)

midfielders['goals_per_shot_on_target'] = np.where(
    midfielders['Shots on target'] > 0,
    midfielders['Goals'] / midfielders['Shots on target'],
    np.nan
)


In [ ]:
#data visualization for midfielders

In [ ]:
#data cleaning for defenders
#filtering defenders with at least 1 appearance
defenders = df_analysis[df_analysis['Position'].str.contains('Defender')&(df_analysis['Appearances'] > 0)]
# Drop the specified columns from the defenders DataFrame
cols_to_drop_defender = [
    'Goals','Goals per match','Headed goals','Goals with right foot','Goals with left foot',
    'Penalties scored','Freekicks scored','Shots','Shots on target','Shooting accuracy %',
    'Hit woodwork','Big chances missed','Assists','Passes','Passes per match',
    'Big chances created','Crosses','Cross accuracy %','Through balls','Accurate long balls'
]

defenders = defenders.drop(columns=cols_to_drop_defender)

#feature engineering 
defenders['tackle_success_rate'] = np.where(
    defenders['Tackles'] > 0,
    defenders['Tackle success %'] / 100,
    np.nan
)

defenders['aerial_duel_success'] = np.where(
    (defenders['Aerial battles won'] + defenders['Aerial battles lost']) > 0,
    defenders['Aerial battles won'] / (defenders['Aerial battles won'] + defenders['Aerial battles lost']),
    np.nan
)

defenders['clearances_per_match'] = defenders['Clearances'] / defenders['Appearances']
defenders['duel_success_rate'] = np.where(
    (defenders['Duels won'] + defenders['Duels lost']) > 0,
    defenders['Duels won'] / (defenders['Duels won'] + defenders['Duels lost']),
    np.nan
)

defenders['errors_per_match'] = defenders['Errors leading to goal'] / defenders['Appearances']

defenders.describe()

In [ ]:
#data visualization for defenders

In [ ]:
#data cleaning for goalkeepers
#filtering goalkeepers with at least 1 appearance
goalkeepers = df_analysis[df_analysis['Position'].str.contains('Goalkeeper')&(df_analysis['Appearances'] > 0)]
# Drop the specified columns from the goalkeepers DataFrame
cols_to_drop_goalkeeper = [
    'Goals','Goals per match','Headed goals','Goals with right foot','Goals with left foot',
    'Penalties scored','Freekicks scored','Shots','Shots on target','Shooting accuracy %',
    'Hit woodwork','Big chances missed','Assists','Passes','Passes per match',
    'Big chances created','Crosses','Cross accuracy %','Through balls','Accurate long balls',
    'Tackles','Tackle success %','Last man tackles','Blocked shots','Interceptions','Clearances',
    'Headed Clearance','Clearances off line','Recoveries','Duels won','Duels lost','Successful 50/50s',
    'Aerial battles won','Aerial battles lost','Own goals','Errors leading to goal','Offsides'
]

goalkeepers = goalkeepers.drop(columns=cols_to_drop_goalkeeper)


#feature engineering 
goalkeepers['save_percentage'] = np.where(
    (goalkeepers['Saves'] + goalkeepers['Goals conceded']) > 0,
    goalkeepers['Saves'] / (goalkeepers['Saves'] + goalkeepers['Goals conceded']),
    np.nan
)

goalkeepers['clean_sheets_per_match'] = goalkeepers['Clean sheets'] / goalkeepers['Appearances']
goalkeepers['goals_conceded_per_match'] = goalkeepers['Goals conceded'] / goalkeepers['Appearances']

goalkeepers['punches_high_claims_per_match'] = (goalkeepers['Punches'] + goalkeepers['High Claims']) / goalkeepers['Appearances']

goalkeepers['penalty_save_rate'] = np.where(
    goalkeepers['Penalties saved'] > 0,
    goalkeepers['Penalties saved'] / goalkeepers['Penalties saved'],  # لو عندك عدد penalties faced replace accordingly
    np.nan
)
goalkeepers.describe()

In [ ]:
#data visualization for midfielders